In [3]:
import time
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os, json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
base = 'C:/Projects/Housing-Price-Prediction/data'
train = pd.read_csv(f'{base}/processed/train_final.csv')
val   = pd.read_csv(f'{base}/processed/val_final.csv')
test  = pd.read_csv(f'{base}/processed/test_final.csv')
print(train.shape, val.shape, test.shape)

(574484, 43) (147885, 43) (149424, 43)


In [5]:
non_features = ['zipcode', 'price', 'year', 'city', 'date']

X_train, y_train = train.drop(columns=non_features), train['price']
X_val,   y_val   = val.drop(columns=non_features),   val['price']
X_test,  y_test  = test.drop(columns=non_features),  test['price']

print([c for c in non_features if c in X_train.columns])          # should be []
print('median_sale_price' in X_train.columns)                      # should be True
print(list(X_train.columns) == list(X_val.columns) == list(X_test.columns))  # should be True
print(X_train.shape[1])                                            # should be 38

[]
True
True
38


In [4]:
def score(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs(y_true - y_pred) / y_true) * 100
    r2   = r2_score(y_true, y_pred)
    return {'RMSE': float(rmse), 'MAE': float(mae), 'MAPE_%': float(mape), 'R2': float(r2)}

In [5]:
with open(f'{base}/results/best_params.json') as f:
    best_params = json.load(f)
print(best_params)

{'iterations': 1399, 'learning_rate': 0.055641006786784464, 'depth': 9, 'l2_leaf_reg': 5.863767210684515, 'random_strength': 4.454153959889799, 'bagging_temperature': 0.29959847099830067, 'border_count': 235}


In [6]:
X_full = pd.concat([X_train, X_val], ignore_index=True)
y_full = pd.concat([y_train, y_val], ignore_index=True)
print(X_full.shape, y_full.shape)

(722369, 38) (722369,)


In [7]:
t0 = time.time()
final_model = CatBoostRegressor(**best_params, random_seed=42, verbose=0,
                                allow_writing_files=False)
final_model.fit(X_full, y_full)
print("fit time (minutes):", round((time.time() - t0) / 60, 1))

fit time (minutes): 4.2


In [8]:
pred_test = final_model.predict(X_test)
print({k: round(v, 2) for k, v in score(y_test, pred_test).items()})
print("mean actual:", round(y_test.mean()), "| mean predicted:", round(float(pred_test.mean())))

{'RMSE': 92092.07, 'MAE': 41534.02, 'MAPE_%': 7.62, 'R2': 0.95}
mean actual: 538031 | mean predicted: 529479


In [9]:
os.makedirs('C:/Projects/Housing-Price-Prediction/models', exist_ok=True)
final_model.save_model('C:/Projects/Housing-Price-Prediction/models/catboost_final.cbm')

In [10]:
app = test[['date', 'city', 'zipcode', 'price']].copy()
app['date'] = pd.to_datetime(app['date'])
app = app.rename(columns={'city': 'region', 'price': 'actual_price'})
app['prediction'] = pred_test
app.to_csv(f'{base}/results/app_table.csv', index=False)
print(app.head())

        date                           region  zipcode   actual_price  \
0 2022-01-31        Orlando-Kissimmee-Sanford    32771  314667.746005   
1 2022-01-31  Tampa-St. Petersburg-Clearwater    33778  325271.243813   
2 2022-01-31   Philadelphia-Camden-Wilmington    21914  278119.378304   
3 2022-01-31                        St. Louis    63380  253267.647544   
4 2022-01-31        San Antonio-New Braunfels    78245  253160.479867   

      prediction  
0  318847.524650  
1  293887.971529  
2  301334.826827  
3  228469.724602  
4  267864.138041  


In [1]:
import joblib
from category_encoders import TargetEncoder

In [6]:
# 1. Same settings as the original: TargetEncoder(cols=['city']), defaults
te_full = TargetEncoder(cols=['city'])
enc_train = te_full.fit_transform(train[['city']], train['price'])

In [7]:
# 2. Does it reproduce the values the model was trained on?
print(np.allclose(enc_train['city'].values, train['city_encoded'].values))   # should be True
print(np.abs(enc_train['city'].values - train['city_encoded'].values).max())

True
0.0


In [8]:
# 3. Do val and test also match?
for name, df in [('val', val), ('test', test)]:
    e = te_full.transform(df[['city']])['city'].values
    print(name, np.allclose(e, df['city_encoded'].values))                   # should be True

val True
test True


In [9]:
# 4. Only if all True: save it
os.makedirs('C:/Projects/Housing-Price-Prediction/artifacts', exist_ok=True)
joblib.dump(te_full, 'C:/Projects/Housing-Price-Prediction/artifacts/city_encoder.joblib')

['C:/Projects/Housing-Price-Prediction/artifacts/city_encoder.joblib']

In [10]:
import sys, pandas, numpy, sklearn, catboost, category_encoders, joblib
print(sys.executable)
print(sys.version)
for m in (pandas, numpy, sklearn, catboost, category_encoders, joblib):
    print(m.__name__, m.__version__)

c:\Users\ghosh\AppData\Local\Programs\Python\Python312\python.exe
3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
pandas 3.1.1
numpy 2.3.5
sklearn 1.8.0
catboost 1.2.10
category_encoders 2.8.1
joblib 1.5.3


In [11]:
import sys
print(sys.executable)
%pip freeze

c:\Users\ghosh\AppData\Local\Programs\Python\Python312\python.exe
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aiosignal==1.4.0
alembic==1.20.1
annotated-doc==0.0.7
annotated-types==0.7.0
anyio==4.16.0
asttokens==3.0.1
attrs==25.4.0
backports.zstd==1.2.0
blinker==1.7.0
brotli==1.2.0
cachetools==7.3.4
catboost==1.2.10
category_encoders==2.9.0
certifi==2026.11.20
cffi==2.0.0
charset-normalizer==3.4.4
click==8.1.7
cloudpickle==3.1.3
colorama==0.4.6
colorlog==6.12.0
comm==0.2.3
contourpy==1.3.3
cryptography==46.0.5
cycler==0.12.1
dacite==1.9.2
dash-core-components==2.0.0
dash-html-components==2.0.0
dash-table==5.0.0
databricks-sdk==0.171.0
debugpy==1.8.19
decorator==5.2.1
docker==7.2.0
et_xmlfile==2.0.0
executing==2.2.1
fastapi==0.165.0
fastjsonschema==2.22.3
filetype==1.2.0
Flask==3.0.0
flask-cors==6.0.5
fonttools==4.61.1
frozenlist==1.8.0
future==1.0.0
gitdb==4.0.12
GitPython==3.2.1
google-ai-generativelanguage==0.6.15
google-api-core==2.30.0
google-api-python-client==2.192.0
google-auth==2.4